# 04 - Barres d'erreur sur les données brutes

Ce notebook reste sur le brut. Il calcule des marges d'erreur approximatives uniquement quand `sample_size` est disponible. Le dataset corrigé sera traité séparément.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

try:
    from IPython.display import display
except ImportError:
    display = print

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
pio.renderers.default = "json"

from presidentielle2027.analytics.uncertainty import approximate_margin_of_error
from presidentielle2027.dashboard.colors import get_political_color
from presidentielle2027.extraction.canonicalization import canonicalize_candidate_fields, is_generic_bloc_label

path = REPO_ROOT / "data" / "processed" / "wikipedia_2027_polls_normalized_v2.csv"
frame = pd.read_csv(path)
canonical = frame.apply(lambda row: canonicalize_candidate_fields(row.get("candidate_name"), row.get("candidate_party"), row.get("political_family")), axis=1, result_type="expand")
canonical.columns = ["candidate_name", "candidate_party", "political_family"]
frame[["candidate_name", "candidate_party", "political_family"]] = canonical
frame["publication_date"] = pd.to_datetime(frame["publication_date"], errors="coerce")
frame["estimate_percent"] = pd.to_numeric(frame["estimate_percent"], errors="coerce")
frame["sample_size"] = pd.to_numeric(frame.get("sample_size"), errors="coerce")
frame = frame.loc[(frame["round"] == "first_round") & (~frame["candidate_name"].map(is_generic_bloc_label))].copy()
scenario_catalog = frame.groupby("scenario_name", dropna=False).agg(rows=("poll_id", "count"), pollsters=("polling_company", "nunique"), last_publication=("publication_date", "max")).sort_values(["rows", "last_publication"], ascending=[False, False])
SCENARIO_NAME = scenario_catalog.index[0]
scenario_frame = frame.loc[frame["scenario_name"] == SCENARIO_NAME].copy()
scenario_frame["approx_moe"] = scenario_frame.apply(lambda row: approximate_margin_of_error(row.get("sample_size"), row.get("estimate_percent", 50.0)), axis=1)
scenario_frame["lower"] = scenario_frame["estimate_percent"] - scenario_frame["approx_moe"].fillna(0.0)
scenario_frame["upper"] = scenario_frame["estimate_percent"] + scenario_frame["approx_moe"].fillna(0.0)
pd.Series({
    "scenario": SCENARIO_NAME,
    "rows": len(scenario_frame),
    "rows_with_error_bars": int(scenario_frame["approx_moe"].notna().sum()),
    "rows_without_error_bars": int(scenario_frame["approx_moe"].isna().sum()),
})

In [ ]:
coverage_by_candidate = scenario_frame.groupby("candidate_name", dropna=False).agg(rows=("poll_id", "count"), rows_with_sample=("approx_moe", lambda s: int(s.notna().sum())), average_moe=("approx_moe", "mean"), average_sample_size=("sample_size", "mean")).sort_values(["rows_with_sample", "rows"], ascending=[False, False])
display(coverage_by_candidate)

scenario_frame[["publication_date", "polling_company", "candidate_name", "estimate_percent", "sample_size", "approx_moe", "source_url"]].sort_values(["publication_date", "candidate_name"], ascending=[False, True]).head(50)

In [ ]:
figure = go.Figure()
for candidate_name, group in scenario_frame.groupby("candidate_name", dropna=False):
    ordered = group.sort_values("publication_date")
    party = ordered["candidate_party"].dropna().iloc[0] if ordered["candidate_party"].notna().any() else None
    family = ordered["political_family"].dropna().iloc[0] if ordered["political_family"].notna().any() else None
    color = get_political_color(party, family)
    figure.add_trace(go.Scatter(x=ordered["publication_date"], y=ordered["estimate_percent"], mode="markers", name=candidate_name, marker={"size": 8, "color": color}, error_y={"type": "data", "array": ordered["upper"] - ordered["estimate_percent"], "arrayminus": ordered["estimate_percent"] - ordered["lower"], "color": color, "thickness": 1.2}))

figure.update_layout(title=f"Premier tour brut avec barres d'erreur approximatives - {SCENARIO_NAME}", xaxis_title="Date de publication", yaxis_title="Intentions de vote (%)", paper_bgcolor="white", plot_bgcolor="white", legend_title="Candidats")
figure.update_xaxes(showgrid=True, gridcolor="#e5e5e5")
figure.update_yaxes(showgrid=True, gridcolor="#e5e5e5")
figure